# 🕵️ Dia 35 — Como o ATT&CK é construído, da tática ao procedimento

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 35* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-35.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Uma semana depois do primeiro SMS, o IscaKit reapareceu contra mais quatro cooperativas: CoopVale, CredMinas, Unicoop Sul e Cooperouro. Você quis sair mapeando tudo no ATT&CK de cabeça. Rafa travou: *"Em que versão? Na matriz Enterprise ou na Mobile? Hoje você não mapeia nada. Hoje você aprende a **consultar**."*

Neste notebook você vai:

1. **Carregar o ATT&CK** (completo, pela rede, ou a amostra enxuta da v19.2 embutida aqui) e ler a versão.
2. **Refazer os passos 2 a 5 do original em Python puro**, sem `jq`.
3. **Gerar o `sub-density-layer.json`** para o Navigator.
4. **Jogar o quiz das quatro camadas** (tática, técnica, subtécnica, procedimento).
5. **Conferir o vocabulário do IscaKit** (Prática BR) e sair com a tabela corrigida.

🐍 Só biblioteca padrão do Python. Nada para instalar. Funciona offline: toda célula que tenta a rede tem plano B embutido. Rode as células de cima para baixo (Shift+Enter).

## 1. Carregar o ATT&CK

O original baixa o arquivo completo (`enterprise-attack.json`, uns 50 MB) e consulta com `jq`. Aqui fazemos o mesmo com `json` e `urllib`, mas com **plano B**: se a rede falhar (ou se você deixar `BAIXAR_COMPLETO = False`), o notebook usa uma **amostra enxuta da v19.2** embutida na próxima célula.

A amostra tem só os campos que as consultas do dia leem: coleção, táticas, todas as técnicas (com `revoked`, `x_mitre_deprecated`, `x_mitre_is_subtechnique` e `kill_chain_phases`), os relacionamentos `revoked-by` e os `uses` de dois grupos (APT29 e APT28). Ela foi extraída do arquivo oficial em 2026-10-12 e **não substitui** o arquivo completo no trabalho real.

> 💡 A célula da amostra é longa (é um texto compactado). Pode rodar sem ler.

In [ ]:
# Amostra enxuta do Enterprise ATT&CK v19.2 (STIX 2.1), compactada com zlib e codificada em base64.
# Por que compactar? Para caber no notebook. Por que base64? Porque JSON compactado é binário,
# e um notebook só guarda texto.
AMOSTRA_B64 = (
    "..."
)
print(f"Amostra embutida: {len(AMOSTRA_B64):,} caracteres em base64.")

In [ ]:
#@title Fonte dos dados { display-mode: "form" }
BAIXAR_COMPLETO = False  #@param {type:"boolean"}
# False = usa a amostra embutida (rápido, funciona offline).
# True  = tenta baixar o arquivo oficial completo (~50 MB). Se falhar, cai na amostra.

import base64          # para desfazer o base64 da amostra
import json            # o ATT&CK é um grande arquivo JSON (um "bundle" STIX)
import urllib.request  # para baixar da internet, sem instalar nada
import zlib            # para descompactar a amostra

URL_COMPLETO = ("https://raw.githubusercontent.com/mitre-attack/attack-stix-data/"
                "master/enterprise-attack/enterprise-attack.json")

def carregar_amostra():
    """Desfaz base64 -> descompacta zlib -> lê JSON. Devolve o bundle STIX."""
    bruto = zlib.decompress(base64.b64decode(AMOSTRA_B64))
    return json.loads(bruto)

bundle, fonte = None, None
if BAIXAR_COMPLETO:
    try:
        # timeout alto: são ~50 MB. Se a rede cair, vamos para o plano B.
        with urllib.request.urlopen(URL_COMPLETO, timeout=120) as resposta:
            bundle = json.load(resposta)
        fonte = "arquivo oficial completo (rede)"
    except Exception as erro:
        print(f"⚠️ Não deu para baixar ({erro.__class__.__name__}). Usando a amostra embutida.")

if bundle is None:
    bundle = carregar_amostra()
    fonte = "amostra enxuta embutida (v19.2, extraída em 2026-10-12)"

OBJ = bundle["objects"]   # todo o conteúdo do ATT&CK é uma lista de objetos STIX

# Passo 1 do original: ler a versão. É a primeira coisa que vai para as suas notas.
colecao = next(o for o in OBJ if o["type"] == "x-mitre-collection")
VERSAO = f'{colecao["name"]} v{colecao["x_mitre_version"]}'
print("Fonte :", fonte)
print("Versão:", VERSAO)
print(f"Objetos carregados: {len(OBJ):,}")

## 2. Contar o que existe na matriz (passo 2 do original)

Duas funções pequenas vão aparecer em todo o resto do notebook:

- `id_externo(obj)`: pega o `T####` / `TA####` que fica escondido dentro de `external_references`;
- `ativo(obj)`: diz se o objeto **não** está revogado nem obsoleto.

O original avisa: objetos revogados e obsoletos **continuam no arquivo**. Se você não filtrar, conta errado.

In [ ]:
def id_externo(obj):
    """Devolve o ID do ATT&CK (ex.: 'T1566.002') de um objeto STIX, ou None."""
    for ref in obj.get("external_references", []):          # a lista de referências do objeto
        if ref.get("source_name") == "mitre-attack" and ref.get("external_id"):
            return ref["external_id"]                       # a referência do MITRE carrega o ID
    return None                                             # objetos sem ID (ex.: relacionamentos)

def ativo(obj):
    """True se o objeto não foi revogado nem marcado como obsoleto."""
    return not obj.get("revoked") and not obj.get("x_mitre_deprecated")

# Separamos as técnicas ativas em mães e subtécnicas, igual às duas consultas jq do original.
tecnicas = [o for o in OBJ if o["type"] == "attack-pattern" and ativo(o)]
maes = [t for t in tecnicas if not t.get("x_mitre_is_subtechnique")]
subs = [t for t in tecnicas if t.get("x_mitre_is_subtechnique")]
taticas = sorted((id_externo(o), o["name"], o["x_mitre_shortname"])
                 for o in OBJ if o["type"] == "x-mitre-tactic")

print(f"Técnicas-mãe: {len(maes)} | Subtécnicas: {len(subs)} | Táticas: {len(taticas)}")
print("(Na v19.2 o original obteve 222, 475 e 15.)\n")
for tid, nome, curto in taticas:
    print(f"{tid}\t{nome:<22}\t{curto}")

## 3. Uma família de técnicas e uma técnica com quatro táticas (passo 3)

O vínculo técnica → tática mora em `kill_chain_phases`. Uma técnica pode ter **várias** fases. Rode para `T1566` e depois para `T1078`.

In [ ]:
#@title Família a consultar { display-mode: "form" }
PREFIXO = "T1566"  #@param {type:"string"}

INDICE = {}                                # ID -> objeto, para consultas rápidas
for o in OBJ:
    if o["type"] == "attack-pattern":
        INDICE.setdefault(id_externo(o), o)

def fases(obj):
    """Lista os nomes de fase (= shortname das táticas) de uma técnica."""
    return [p["phase_name"] for p in obj.get("kill_chain_phases", [])]

familia = sorted((id_externo(t), t["name"], ",".join(fases(t)))
                 for t in OBJ if t["type"] == "attack-pattern" and not t.get("revoked")
                 and (id_externo(t) or "").startswith(PREFIXO.strip()))
for linha in familia:
    print("\t".join(linha))

print("\nE a T1078, que mora em quatro colunas:")
print("T1078", INDICE["T1078"]["name"], "→", fases(INDICE["T1078"]))

## 4. Resolver um ID revogado (passo 4)

Em `jq`, o original faz isso em duas consultas. Em Python, é uma função: acha o objeto antigo, procura um relacionamento `revoked-by` saindo dele e devolve o alvo. Se o substituto também tiver sido revogado (acontece), seguimos a corrente.

In [ ]:
POR_STIX_ID = {o["id"]: o for o in OBJ}     # id STIX -> objeto
REVOGADO_POR = {o["source_ref"]: o["target_ref"]  # id antigo -> id novo
                for o in OBJ if o["type"] == "relationship" and o.get("relationship_type") == "revoked-by"}

def resolver(tid):
    """Recebe um ID do ATT&CK e devolve (id_final, nome, caminho percorrido)."""
    obj = INDICE.get(tid)
    if obj is None:
        return None, "não existe neste conjunto de dados", [tid]
    caminho = [tid]
    while obj.get("revoked") and obj["id"] in REVOGADO_POR:   # segue a corrente de revogações
        obj = POR_STIX_ID[REVOGADO_POR[obj["id"]]]
        caminho.append(id_externo(obj))
    return id_externo(obj), obj["name"], caminho

for antigo in ["T1070.001", "T1562.008", "T1566.002"]:
    final, nome, caminho = resolver(antigo)
    status = "revogado" if len(caminho) > 1 else "sem mudança"
    print(f"{antigo:<10} → {final:<10} {nome}  ({status}: {' → '.join(caminho)})")

## 5. Gerar a camada de densidade de subtécnicas (passo 5)

O mesmo `sublayer.jq` do original, em Python. O arquivo sai pronto para o Navigator (formato de camada 4.5).

In [ ]:
from collections import Counter   # conta ocorrências sem escrever laço de contagem

ids_ativos = [id_externo(t) for t in tecnicas]
# Para cada subtécnica "T1027.006", contamos +1 para a mãe "T1027".
contagem = Counter(i.split(".")[0] for i in ids_ativos if "." in i)

camada = {
    "name": "Sub-technique density",
    "versions": {"layer": "4.5", "attack": colecao["x_mitre_version"].split(".")[0], "navigator": "5.3.2"},
    "domain": "enterprise-attack",
    "description": f"Score = number of active sub-techniques under each parent technique ({VERSAO})",
    "techniques": [{"techniqueID": i, "score": contagem.get(i, 0)} for i in ids_ativos if "." not in i],
    "gradient": {"colors": ["#ffffff", "#ff6666"], "minValue": 0, "maxValue": 10},
}
with open("sub-density-layer.json", "w", encoding="utf-8") as f:
    json.dump(camada, f, indent=1)

top = sorted(camada["techniques"], key=lambda t: -t["score"])[:5]
print("Top 5:", [(t["techniqueID"], t["score"]) for t in top])
print("(Na v19.2: T1027 e T1546 com 18; T1547, T1564 e T1218 com 14.)")
print("💾 sub-density-layer.json gerado. Suba no Navigator: Open Existing Layer → Upload from local.")

## 6. Ficha de IDs: procedimento vem do dado (passo 6)

O texto de procedimento é a `description` de um relacionamento `uses`. A amostra embutida guarda as descrições dos `uses` de APT29 e APT28 para cinco técnicas; com o arquivo completo, qualquer técnica funciona.

In [ ]:
def procedimentos(tid, limite=3):
    """Devolve até `limite` textos de procedimento (relacionamentos uses) para uma técnica."""
    alvo = INDICE[tid]["id"]
    textos = []
    for o in OBJ:
        if o["type"] == "relationship" and o.get("relationship_type") == "uses" \
                and o.get("target_ref") == alvo and o.get("description"):
            quem = POR_STIX_ID.get(o["source_ref"], {}).get("name", "?")
            textos.append((quem, o["description"]))
    return textos[:limite]

for tid in ["T1566.002", "T1110.003", "T1090.004"]:
    print(f"\n{tid} {INDICE[tid]['name']} — táticas: {fases(INDICE[tid])}")
    for quem, texto in procedimentos(tid):
        print(f"   [{quem}] {texto[:220]}")

## 7. 🎮 Quiz: tática, técnica, subtécnica ou procedimento?

Oito itens. Para cada um, responda **1** (tática), **2** (técnica), **3** (subtécnica) ou **4** (procedimento). Quer só ler o gabarito? `MODO_JOGO = False`.

In [ ]:
MODO_JOGO = True   # False = só mostra as respostas

CAMADAS = {"1": "tática", "2": "técnica", "3": "subtécnica", "4": "procedimento"}
QUIZ = [
    ("TA0001 Initial Access", "1", "Responde POR QUÊ: o objetivo naquela etapa."),
    ("T1566 Phishing", "2", "COMO, em termos gerais. Formato T####."),
    ("T1566.002 Spearphishing Link", "3", "COMO, mais específico. Formato T####.###."),
    ("'APT29 has used spearphishing links...' (com citação)", "4", "Texto de um relacionamento uses, sem ID próprio."),
    ("TA0005 Stealth", "1", "Tática. Desde a v19.0 ficou com o TA0005 da antiga Defense Evasion."),
    ("'O IscaKit mandou SMS com link para serraazul-pix.example em 5/out' (relatório da Vigia)", "4",
     "É o que um ator específico fez, com fonte. Isso é procedimento."),
    ("T1660 Phishing (matriz Mobile)", "2", "Técnica do domínio Mobile. Mesmo nome, outro domínio, outro ID."),
    ("T1056.003 Web Portal Capture", "3", "Subtécnica de T1056 Input Capture."),
]

acertos = 0
for n, (item, certo, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{n}/{len(QUIZ)}] {item}")
    if MODO_JOGO:
        resp = ""
        while resp not in CAMADAS:
            resp = input("   1=tática 2=técnica 3=subtécnica 4=procedimento: ").strip()
        acertos += (resp == certo)
        print("   ✅ Isso!" if resp == certo else f"   ❌ Era {CAMADAS[certo]}.")
    print(f"   → {CAMADAS[certo]}: {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 8. ➕ Prática BR: o vocabulário do IscaKit, conferido no dado

A tabela de rascunho que você escreveu de memória na abertura. A célula confere cada palpite no dataset carregado (Enterprise) e numa **mini-tabela da matriz Mobile v19.2** embutida (extraída do `mobile-attack.json` oficial em 2026-10-12). Repare em quem está revogado.

In [ ]:
# Rascunho de memória: (comportamento, palpite)
RASCUNHO = [
    ("SMS com link para o site falso", "T1566.002"),
    ("Página falsa que coleta a senha", "T1056.003"),
    ("Registro dos domínios imitação", "T1583.001"),
    ("Certificado TLS gratuito na véspera", "T1588.004"),
    ("Encurtador lnk.example/sa7 apontando para o site", "T1608.005"),
    ("Mensagem que se passa pela cooperativa", "T1656"),
    ("Transferências Pix não reconhecidas", "T1657"),
]

# Mini-tabela do Mobile ATT&CK v19.2 (ID, nome, táticas). Fonte: mobile-attack.json oficial.
MOBILE_V19_2 = {
    "T1660": ("Phishing", ["initial-access"]),
    "T1417.002": ("GUI Input Capture", ["credential-access", "collection"]),
    "T1655": ("Masquerading", ["defense-evasion"]),
}
TATICAS_MOBILE = {"initial-access": "TA0027", "credential-access": "TA0031",
                  "collection": "TA0035", "defense-evasion": "TA0030"}

linhas_md = ["| Comportamento | Palpite | ID conferido | Nome oficial | Versão / domínio |", "|---|---|---|---|---|"]
for comportamento, palpite in RASCUNHO:
    final, nome, caminho = resolver(palpite)
    nota = f"{VERSAO.split(' v')[-1]} / Enterprise"
    if len(caminho) > 1:
        nota += f" (⚠️ {palpite} revogado → {final})"
    print(f"{palpite:<10} → {final or '???':<10} {nome}   [{' → '.join(caminho)}]")
    linhas_md.append(f"| {comportamento} | {palpite} | {final} | {nome} | {nota} |")

print("\nE a casa do SMS na matriz Mobile:")
for tid, (nome, fs) in MOBILE_V19_2.items():
    print(f"   {tid:<10} {nome:<20} táticas Mobile: {[TATICAS_MOBILE[x] + ' ' + x for x in fs]}")
linhas_md.append("| SMS com link (alternativa Mobile) | — | T1660 | Phishing | 19.2 / Mobile (tática TA0027) |")

with open("iscakit-vocabulario-conferido.md", "w", encoding="utf-8") as f:
    f.write("# IscaKit — vocabulário ATT&CK conferido\n\n" + "\n".join(linhas_md) + "\n")
print("\n💾 iscakit-vocabulario-conferido.md salvo.")

> 🧠 **Camada 8.** Qual linha você teria jurado que estava certa? A fluência (ter lido "T1656" em dezenas de relatórios) não avisa quando um ID é revogado. A sensação de "isso eu sei" é o seu gatilho para conferir.

In [ ]:
#@title Sua decisão: Enterprise ou Mobile para o disparo do SMS? { display-mode: "form" }
DOMINIO_SMS = "Mobile (T1660), com T1566.002 como referência cruzada"  #@param ["Mobile (T1660), com T1566.002 como referência cruzada", "Enterprise (T1566.002)", "Os dois, até a revisão do Dia 43"]
JUSTIFICATIVA = "A mensagem chegou ao celular pessoal do cooperado, fora de qualquer sistema corporativo; a matriz Mobile descreve exatamente esse vetor."  #@param {type:"string"}

nota = f"""# Notas do Dia 35
Versão do ATT&CK: {VERSAO} (fonte: {fonte})
ID revogado resolvido (passo 4): T1070.001 → {resolver('T1070.001')[0]}
ID de relatório antigo (passo 7 / Prática BR): T1656 → {resolver('T1656')[0]}
Domínio para o SMS: {DOMINIO_SMS}
Por quê: {JUSTIFICATIVA}
"""
with open("notas-dia35.md", "w", encoding="utf-8") as f:
    f.write(nota)
print(nota)

try:                                    # no Colab, baixa os artefatos do dia
    from google.colab import files
    for nome in ["sub-density-layer.json", "iscakit-vocabulario-conferido.md", "notas-dia35.md"]:
        files.download(nome)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 35

- [ ] Versão do ATT&CK anotada (e a fonte: completo ou amostra)
- [ ] `sub-density-layer.json` carregado no Navigator e exportado em SVG
- [ ] Ficha de IDs com cinco linhas, cada célula vinda do dado
- [ ] Tabela do IscaKit conferida, com a linha revogada resolvida
- [ ] Decisão Enterprise × Mobile para o SMS, justificada em duas frases

**Amanhã (Dia 36):** mapear um alerta publicado frase por frase, com uma tabela de evidências, e repetir o método no "comunicado de alerta" da Serra Azul para as outras quatro cooperativas.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>